In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# ---------------------------------------------------------
# 1. Load Data from Previous Step (Task A Output)
# ---------------------------------------------------------
input_file = 'final_credit_risk_predictions.xlsx'
df = pd.read_excel(input_file)

# ---------------------------------------------------------
# 2. Preprocess & Encode Features for Regression
# ---------------------------------------------------------
target_col = 'Default payment next month'

# Clean ratios if present
ratio_cols = [
    'credit utilization ratio', 'pay_to_bill_ratio',
    'mean_monthly_bill', 'mean_monthly_payment', 'max_payment_delay'
]
existing_ratio_cols = [col for col in ratio_cols if col in df.columns]
df[existing_ratio_cols] = df[existing_ratio_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# One-Hot Encoding categorical features
categorical_cols = [col for col in ['EDUCATION', 'Gender', 'MARRIAGE'] if col in df.columns]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# ---------------------------------------------------------
# 3. TASK B: Safe Credit Limit Recommendation (Linear Regression)
# ---------------------------------------------------------
# Exclude non-predictive, target, and Task A outputs from features
exclude_reg = ['ID', 'LIMIT_BAL', target_col, 'Default_Risk_Score', 'Risk_Tier']
X_reg = df_encoded.drop(columns=exclude_reg, errors='ignore')
y_reg = df_encoded['LIMIT_BAL']

# Train Linear Regression ONLY on non-defaulters
non_default_mask = df_encoded[target_col] == 0
X_train_reg = X_reg[non_default_mask]
y_train_reg = y_reg[non_default_mask]

lin_reg_model = LinearRegression()
lin_reg_model.fit(X_train_reg, y_train_reg)

# Predict statistical baseline credit limit for ALL accounts
df['Recommended_Credit_Limit'] = lin_reg_model.predict(X_reg)

# Standard Rounding (Nearest $1,000) & Floor of $10,000
df['Recommended_Credit_Limit'] = df['Recommended_Credit_Limit'].apply(
    lambda x: max(round(x, -3), 10000)
)

# ---------------------------------------------------------
# 4. Risk Governance Rule: Cap High Risk Accounts at $10,000
# ---------------------------------------------------------
def apply_risk_policy(row):
    if row['Risk_Tier'] == 'High Risk':
        return 10000  # Enforce strictly $10,000 cap for High Risk customers
    else:
        return row['Recommended_Credit_Limit']

df['Final_Approved_Limit'] = df.apply(apply_risk_policy, axis=1)

# ---------------------------------------------------------
# 5. Export Final Enriched Dataset to Excel
# ---------------------------------------------------------
output_file = 'final_credit_risk_predictions_with_limits.xlsx'
df.to_excel(output_file, index=False)

print("=== TASK B & RISK GOVERNANCE COMPLETE ===")
print(f"Loaded source file: '{input_file}'")
print(f"High-risk $10,000 cap applied. Output saved to: '{output_file}'")

#download the excel file
from google.colab import files
# Replace with your file's name
file_name = 'final_credit_risk_predictions_with_limits.xlsx'
# Trigger the browser download
files.download(file_name)

=== TASK B & RISK GOVERNANCE COMPLETE ===
Loaded source file: 'final_credit_risk_predictions.xlsx'
High-risk $10,000 cap applied. Output saved to: 'final_credit_risk_predictions_with_limits.xlsx'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>